# DACON ETRI demographic/static-feature strict audit runner

Report-only Kaggle runner. It performs no DACON submission and creates no submission CSV. The goal is to run the V1 demographic/static-feature strict audit away from local memory pressure.

In [ ]:

import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path


def run(cmd: str) -> None:
    print(f"\n$ {cmd}", flush=True)
    subprocess.run(cmd, shell=True, check=True)

is_kaggle = Path('/kaggle/input').exists()
work_dir = Path('/kaggle/working') if is_kaggle else Path.cwd()
project_root = Path('/tmp/dacon_project') if is_kaggle else Path.cwd()
output_root = work_dir / 'kaggle_outputs' if is_kaggle else Path('kaggle_outputs')
os.environ['KAGGLE_OUTPUT_DIR'] = str(output_root)

required_payload_paths = [
    Path('scripts/diagnose_strict_demographic_feature_audit.py'),
    Path('src/features/demographics.py'),
    Path('data/processed/df_train_raw.parquet'),
    Path('data/processed/df_test_raw.parquet'),
]

def payload_score(root: Path) -> int:
    return sum((root / rel).exists() for rel in required_payload_paths)

if is_kaggle:
    print('Detected Kaggle environment; locating project payload...')
    source_zips = sorted(Path('/kaggle/input').glob('**/source.zip'), key=lambda p: p.stat().st_mtime, reverse=True)
    project_root.mkdir(parents=True, exist_ok=True)
    if source_zips:
        with zipfile.ZipFile(source_zips[0]) as zf:
            zf.extractall(project_root)
        print(f'Extracted {source_zips[0]} -> {project_root}')
    else:
        payload_roots = []
        for root, dirs, files in os.walk('/kaggle/input'):
            if {'src', 'conf', 'scripts'}.issubset(set(dirs)):
                payload_roots.append(Path(root))
        if not payload_roots:
            raise FileNotFoundError('No source.zip or unpacked project root with src/conf/scripts found in Kaggle inputs.')
        payload_root = sorted(payload_roots, key=lambda x: (-payload_score(x), len(str(x))))[0]
        print(f'Found unpacked payload at: {payload_root}')
        for name in ['src', 'conf', 'scripts', 'data', 'artifacts', 'submissions', 'pyproject.toml', 'README.md']:
            src_path = payload_root / name
            dst_path = project_root / name
            if not src_path.exists():
                continue
            if src_path.is_dir():
                shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
            else:
                dst_path.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src_path, dst_path)
else:
    print('Local dry-run mode; using current repository root.')

os.chdir(project_root)
sys.path.insert(0, str(project_root))
for folder in ['data/processed', 'reports', 'submissions', 'artifacts/base_predictions', 'notebooks/history']:
    Path(folder).mkdir(parents=True, exist_ok=True)
output_root.mkdir(parents=True, exist_ok=True)
missing_required = [str(path) for path in required_payload_paths if not path.exists()]
if missing_required:
    print('Top-level project files:', sorted(str(p) for p in Path('.').iterdir())[:40])
    print('scripts sample:', sorted(str(p) for p in Path('scripts').glob('*.py'))[:40] if Path('scripts').exists() else [])
    raise FileNotFoundError(f'Missing demographic audit payload files: {missing_required}')
print('project_root =', project_root)
print('output_root =', output_root)
print('required payload files verified:', [str(path) for path in required_payload_paths])


In [ ]:

# Keep installs explicit and CPU-friendly. Kaggle usually already has these.
if is_kaggle:
    import importlib.util
    required = {
        'sklearn': 'scikit-learn',
        'pandas': 'pandas',
        'numpy': 'numpy',
        'pyarrow': 'pyarrow',
        'nbformat': 'nbformat',
    }
    missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
    print('missing packages =', missing)
    if missing:
        run(f"{sys.executable} -m pip install -q {' '.join(missing)}")


In [ ]:

# Report-only V1 demographic/static-feature strict audit.
import time
from pathlib import Path

RUN_STARTED_TS = time.time()
STRICT_EXPERIMENT_COMMAND = (
    f"{sys.executable} scripts/diagnose_strict_demographic_feature_audit.py "
    "--output-json reports/strict_demographic_feature_audit_kaggle_20260617.json "
    "--n-estimators 240"
)
run(STRICT_EXPERIMENT_COMMAND)


In [ ]:

# Stage report artifacts for `kaggle kernels output`.
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

out_dir = Path(os.environ.get('KAGGLE_OUTPUT_DIR', 'kaggle_outputs'))
out_dir.mkdir(parents=True, exist_ok=True)
patterns = [
    'reports/strict_demographic_feature_audit_kaggle_20260617.json',
    'reports/strict_demographic_feature_audit_kaggle_20260617.md',
]
copied = []
for pattern in patterns:
    for path in sorted(Path('.').glob(pattern)):
        if path.is_file():
            dest = out_dir / path.name
            shutil.copy2(path, dest)
            copied.append(str(path))
manifest = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'command': STRICT_EXPERIMENT_COMMAND,
    'copied_artifacts': copied,
    'note': 'Report-only demographic/static-feature strict audit. No DACON submission was made.',
}
(out_dir / 'remote_demographic_audit_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(manifest, ensure_ascii=False, indent=2))
